## ML Feasibility Assessment

`It is designed to determine whether Machine Learning can provide a practical solution to the financial-security problem identified through data cleaning and exploratory analysis.`

A business does not benefit from Machine Learning simply because a dataset contains thousands of records. Before investing in a predictive system, a Data Scientist must first establish whether the available data is sufficiently large, structured, reliable, and appropriately labeled for machine learning.

Therefore, the final objective is not to build a model yet, but to answer a more fundamental business question:

___"Does this dataset contain enough reliable and relevant information to justify building a Machine Learning-based fraud detection system?"___

### Exploring DATA:

`A company has millions of transactions and a column called "is_fraud". Does that automatically mean Machine Learning should be used?`

No.

A dataset can be huge and still be unsuitable for ML. It might contain unreliable labels, too few useful features, excessive missing information, duplicated records, or variables that reveal the answer directly.

Similarly, a relatively simple dataset can be highly suitable for Machine Learning if it contains:

- a meaningful prediction target,
+ sufficient observations,
* useful predictive features,
+ consistent structure,
* and a business problem where prediction provides value.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

train_path = Path("../Datasets_MLmodels/Fraud/P03_outputs/fraudTrain_clean.csv")
test_path = Path("../Datasets_MLmodels/Fraud/P03_outputs/fraudTest_clean.csv")

eda_path = Path("../Datasets_MLmodels/Fraud/P04_outputs/fraud_eda_dataset.csv")

output_dir = Path("../Datasets_MLmodels/Fraud/P06_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
print("=" * 75)
print("\tPROJECT 06 - MACHINE LEARNING FEASIBILITY ASSESSMENT")
print("=" * 75)

print("""
Business Domain:
Credit Card Fraud Detection

Objective:
Determine whether Machine Learning is appropriate for detecting
fraudulent transactions using the available simulated transaction data.
""")

	PROJECT 06 - MACHINE LEARNING FEASIBILITY ASSESSMENT

Business Domain:
Credit Card Fraud Detection

Objective:
Determine whether Machine Learning is appropriate for detecting
fraudulent transactions using the available simulated transaction data.



In [3]:
if not train_path.exists():
    raise FileNotFoundError(
        f"Project 03 training output not found:\n{train_path}")

if not test_path.exists():
    raise FileNotFoundError(
        f"Project 03 testing output not found:\n{test_path}")

print("\nProject 03 training dataset found at:")
print(train_path)

print("\nProject 03 testing dataset found at:")
print(test_path)

if eda_path.exists():
    print(f"\nProject 04 EDA dataset found at:\n{eda_path}\n")
else:
    print(
        "\nProject 04 EDA dataset was not found."
        "\nThe feasibility assessment will continue using Project 03 outputs."
    )
print('=' * 70)


Project 03 training dataset found at:
..\Datasets_MLmodels\Fraud\P03_outputs\fraudTrain_clean.csv

Project 03 testing dataset found at:
..\Datasets_MLmodels\Fraud\P03_outputs\fraudTest_clean.csv

Project 04 EDA dataset found at:
..\Datasets_MLmodels\Fraud\P04_outputs\fraud_eda_dataset.csv



In [4]:
fraud_train = pd.read_csv(train_path)
fraud_test = pd.read_csv(test_path)
    
print("Cleaned Datasets loaded successfully!")   

Cleaned Datasets loaded successfully!


In [5]:
if eda_path.exists():
    fraud_eda = pd.read_csv(eda_path)
    print("\nProject 04 analytical dataset loaded successfully.")
    print("EDA dataset shape:", fraud_eda.shape)
else:
    fraud_eda = None


Project 04 analytical dataset loaded successfully.
EDA dataset shape: (1852394, 32)


In [6]:
print("\n" + "=" * 60)
print("\t\t1. DATASET SIZE ASSESSMENT")
print("=" * 60)

train_rows, train_columns = fraud_train.shape
test_rows, test_columns = fraud_test.shape

total_rows = train_rows + test_rows

print("\nTraining observations:", train_rows)
print("Training features:", train_columns)

print("\nTesting observations:", test_rows)
print("Testing features:", test_columns)

print("\nTotal observations:", total_rows)

if total_rows >= 100000:
    size_assessment = "Very large dataset — highly suitable for ML experimentation."
elif total_rows >= 10000:
    size_assessment = "Large dataset — suitable for ML experimentation."
elif total_rows >= 1000:
    size_assessment = "Moderate dataset — ML is feasible with appropriate validation."
else:
    size_assessment = "Small dataset — ML may be possible but requires caution."

print("\nAssessment:")
print(size_assessment)


		1. DATASET SIZE ASSESSMENT

Training observations: 1296675
Training features: 22

Testing observations: 555719
Testing features: 22

Total observations: 1852394

Assessment:
Very large dataset — highly suitable for ML experimentation.


In [7]:
print("\n" + "=" * 80)
print("\t\t\t2. DATA STRUCTURE ASSESSMENT")
print("=" * 80)

print("\nTraining data types:")
print(fraud_train.dtypes)

print("\n\nNumerical columns:")

numeric_columns = fraud_train.select_dtypes(
    include=np.number
).columns.tolist()

print(numeric_columns)

print("\nCategorical columns:")
categorical_columns = fraud_train.select_dtypes(
    include="object"
).columns.tolist()

categorical_columns


			2. DATA STRUCTURE ASSESSMENT

Training data types:
trans_date_trans_time     object
cc_num                     int64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
dtype: object


Numerical columns:
['cc_num', 'amt', 'zip', 'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']

Categorical columns:


['trans_date_trans_time',
 'merchant',
 'category',
 'first',
 'last',
 'gender',
 'street',
 'city',
 'state',
 'job',
 'dob',
 'trans_num']

In [8]:
print("\n" + "=" * 36)
print("3. TARGET VARIABLE ASSESSMENT")
print("=" * 36)

target = "is_fraud"

if target not in fraud_train.columns:
    raise KeyError(
        "Target variable 'is_fraud' is not present."
    )

print("\nTarget variable:", target)

print("\nTarget data type:")
print(fraud_train[target].dtype)

print("\nTarget values:")
print(fraud_train[target].value_counts(dropna=False))

print("\nTarget percentages:")
print(fraud_train[target].value_counts(normalize=True, dropna=False).mul(100).round(2))


3. TARGET VARIABLE ASSESSMENT

Target variable: is_fraud

Target data type:
int64

Target values:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Target percentages:
is_fraud
0    99.42
1     0.58
Name: proportion, dtype: float64


In [9]:
valid_target_values = {0, 1}

actual_target_values = set(fraud_train[target].dropna().unique())
invalid_target_values = (actual_target_values - valid_target_values)

print("\nInvalid target values:")

if invalid_target_values:
    print(invalid_target_values)
else:
    print("None")


Invalid target values:
None


In [10]:
print("\n" + "=" * 80)
print("\t\t\t   4. CLASS BALANCE ASSESSMENT")
print("=" * 80)

class_distribution = (fraud_train[target].value_counts().sort_index())

class_percentages = (fraud_train[target].value_counts(
    normalize=True).sort_index().mul(100)
)
class_summary = pd.DataFrame({
    "transaction_count": class_distribution,
    "percentage": class_percentages.round(2)
})

print(class_summary)

minority_percentage = (class_percentages.min())

print(
    "\nMinority class percentage:",
    round(minority_percentage, 2),
    "%"
)
if minority_percentage < 5:
    imbalance_assessment = (
        "Severe class imbalance detected. "
        "Special evaluation and modeling strategies will be required."
    )
elif minority_percentage < 20:
    imbalance_assessment = (
        "Moderate class imbalance detected. "
        "Stratified validation and appropriate metrics are recommended."
    )
else:
    imbalance_assessment = (
        "Class distribution is relatively balanced."
    )
print("\nAssessment:")
print(imbalance_assessment)


			   4. CLASS BALANCE ASSESSMENT
          transaction_count  percentage
is_fraud                               
0                   1289169       99.42
1                      7506        0.58

Minority class percentage: 0.58 %

Assessment:
Severe class imbalance detected. Special evaluation and modeling strategies will be required.


In [11]:
print("\n" + "=" * 60)
print("\t5. MISSING VALUE FEASIBILITY CHECK")
print("=" * 60)

missing_values = (
    fraud_train
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

missing_values = missing_values[missing_values > 0]

if len(missing_values) == 0:
    print(
        "\nNo missing values remain in the cleaned training dataset."
    )
    missing_assessment = ("Missing-value quality is suitable for ML preparation.")
else:
    print("\nRemaining missing values:")
    print(missing_values)
    missing_assessment = ("Additional missing-value handling is required before modeling.")

print("\nAssessment:")
print(missing_assessment)


	5. MISSING VALUE FEASIBILITY CHECK

No missing values remain in the cleaned training dataset.

Assessment:
Missing-value quality is suitable for ML preparation.


In [12]:
print("\n" + "=" * 40)
print("6. DUPLICATE RECORD ASSESSMENT")
print("=" * 40)

duplicate_train = fraud_train.duplicated().sum()
duplicate_test = fraud_test.duplicated().sum()

print("\nTraining duplicate rows:", duplicate_train)

print("Testing duplicate rows:", duplicate_test)

if duplicate_train == 0 and duplicate_test == 0:
    duplicate_assessment = ("No exact duplicate rows remain.")
else:
    duplicate_assessment = ("Duplicate records remain and should be investigated.")

print("\nAssessment:")
print(duplicate_assessment)


6. DUPLICATE RECORD ASSESSMENT

Training duplicate rows: 0
Testing duplicate rows: 0

Assessment:
No exact duplicate rows remain.


In [13]:
print("\n" + "=" * 60)
print("\t7. FEATURE UNIQUENESS ASSESSMENT")
print("=" * 60)

unique_summary = pd.DataFrame({
    "unique_values": fraud_train.nunique(),
    "unique_percentage": (
        fraud_train.nunique()
        / len(fraud_train)
        * 100
    ).round(2)
})

print(unique_summary.sort_values("unique_percentage", ascending=False))


	7. FEATURE UNIQUENESS ASSESSMENT
                       unique_values  unique_percentage
trans_num                    1296675             100.00
merch_long                   1275745              98.39
trans_date_trans_time        1274791              98.31
unix_time                    1274823              98.31
merch_lat                    1247805              96.23
amt                            52928               4.08
street                           983               0.08
cc_num                           983               0.08
lat                              968               0.07
dob                              968               0.07
city_pop                         879               0.07
long                             969               0.07
zip                              970               0.07
city                             894               0.07
merchant                         693               0.05
job                              494               0.04
last         

In [14]:
high_cardinality_columns = (
    unique_summary[
        unique_summary["unique_percentage"] > 50
    ]
    .index
    .tolist()
)

print("\nHigh-cardinality columns:")
print(high_cardinality_columns)


High-cardinality columns:
['trans_date_trans_time', 'trans_num', 'unix_time', 'merch_lat', 'merch_long']


In [15]:
print("\n" + "=" * 38)
print("8. IDENTIFIER FEATURE INVESTIGATION")
print("=" * 38)

identifier_candidates = ["cc_num", "trans_num"]

for column in identifier_candidates:
    if column in fraud_train.columns:
        unique_count = fraud_train[column].nunique()
        total_count = len(fraud_train)

        uniqueness_ratio = (
            unique_count / total_count
        )
        print(f"\nColumn: {column}")
        print("Unique values:", unique_count)
        print("Uniqueness ratio:", round(uniqueness_ratio, 4))


8. IDENTIFIER FEATURE INVESTIGATION

Column: cc_num
Unique values: 983
Uniqueness ratio: 0.0008

Column: trans_num
Unique values: 1296675
Uniqueness ratio: 1.0


In [16]:
print("\n" + "=" * 70)
print("\t\t9. POTENTIAL DATA LEAKAGE ASSESSMENT")
print("=" * 70)

print("""
Data leakage occurs when information that would not genuinely
be available at prediction time is used to predict the target.

Potentially problematic columns are therefore investigated
before ML modeling.
""")
leakage_candidates = [
    "is_fraud",
    "trans_num",
    "unix_time"
]

for column in leakage_candidates:
    if column in fraud_train.columns:
        print(f"\n{column}: present")
print("""
Interpretation:

- is_fraud is the target and must not be used as an input feature.
- trans_num is an identifier and is unlikely to provide meaningful
  predictive information.
- unix_time requires careful consideration because temporal variables
  can introduce unrealistic assumptions if handled incorrectly.
""")


		9. POTENTIAL DATA LEAKAGE ASSESSMENT

Data leakage occurs when information that would not genuinely
be available at prediction time is used to predict the target.

Potentially problematic columns are therefore investigated
before ML modeling.


is_fraud: present

trans_num: present

unix_time: present

Interpretation:

- is_fraud is the target and must not be used as an input feature.
- trans_num is an identifier and is unlikely to provide meaningful
  predictive information.
- unix_time requires careful consideration because temporal variables
  can introduce unrealistic assumptions if handled incorrectly.



In [17]:
print("\n" + "=" * 65)
print("10. \t\t FEATURE TYPE SUITABILITY")
print("=" * 65)

feature_assessment = []
for column in fraud_train.columns:
    if column == target:
        role = "Target"
        recommendation = "Use as prediction target."
    elif column in ["trans_num", "cc_num"]:
        role = "Identifier"
        recommendation = ("Investigate carefully; generally exclude from model inputs.")
    elif fraud_train[column].dtype == "object":
        role = "Categorical"
        recommendation = ("Can be encoded using an appropriate categorical encoding method.")
   
    elif np.issubdtype(fraud_train[column].dtype, np.number):
        role = "Numerical"
        recommendation = ("Potential numerical ML feature after validation.")
    else:
        role = "Other"
        recommendation = ("Requires additional preprocessing.")

    feature_assessment.append({
        "column": column,
        "role": role,
        "recommendation": recommendation
    })
feature_assessment_df = pd.DataFrame(feature_assessment) 

feature_assessment_df


10. 		 FEATURE TYPE SUITABILITY


,column,role,recommendation
0,trans_date_trans_time,Categorical,Can be encoded using an appropriate categorica...
1,cc_num,Identifier,Investigate carefully; generally exclude from ...
2,merchant,Categorical,Can be encoded using an appropriate categorica...
3,category,Categorical,Can be encoded using an appropriate categorica...
4,amt,Numerical,Potential numerical ML feature after validation.
5,first,Categorical,Can be encoded using an appropriate categorica...
6,last,Categorical,Can be encoded using an appropriate categorica...
7,gender,Categorical,Can be encoded using an appropriate categorica...
8,street,Categorical,Can be encoded using an appropriate categorica...
9,city,Categorical,Can be encoded using an appropriate categorica...


In [18]:
excluded_columns = [target, "trans_num", "cc_num"]

candidate_features = [
    column
    for column in fraud_train.columns
    if column not in excluded_columns
]

print("\n" + "=" * 77)
print("\t\t\t 11. CANDIDATE FEATURE SET")
print("=" * 77)

print("\nCandidate features:")
display(candidate_features)

print("\nNumber of candidate features:", len(candidate_features))


			 11. CANDIDATE FEATURE SET

Candidate features:


['trans_date_trans_time',
 'merchant',
 'category',
 'amt',
 'first',
 'last',
 'gender',
 'street',
 'city',
 'state',
 'zip',
 'lat',
 'long',
 'city_pop',
 'job',
 'dob',
 'unix_time',
 'merch_lat',
 'merch_long']


Number of candidate features: 19


In [19]:
print("\n" + "=" * 40)
print("12. TRAIN / TEST STRUCTURE CHECK")
print("=" * 40)

train_columns = set(fraud_train.columns)
test_columns = set(fraud_test.columns)

only_train = train_columns - test_columns
only_test = test_columns - train_columns

print("\nColumns only in training:")
print(
    only_train
    if only_train
    else "None"
    )

print("\nColumns only in testing:")
print(
    only_test
    if only_test
    else "None"
)


12. TRAIN / TEST STRUCTURE CHECK

Columns only in training:
None

Columns only in testing:
None


In [20]:
print("\n" + "=" * 50)
print("       13. TRAIN / TEST TARGET COMPARISON")
print("=" * 50)

train_fraud_rate = (fraud_train[target].mean() * 100)
test_fraud_rate = (fraud_test[target].mean() * 100)

print("Training fraud rate:", round(train_fraud_rate, 2), "%")
print("Testing fraud rate:", round(test_fraud_rate, 2), "%")

fraud_rate_difference = abs( train_fraud_rate - test_fraud_rate)
print("Absolute difference:", round(fraud_rate_difference, 2), "percentage points")


       13. TRAIN / TEST TARGET COMPARISON
Training fraud rate: 0.58 %
Testing fraud rate: 0.39 %
Absolute difference: 0.19 percentage points


In [21]:
print("\n" + "=" * 45)
print("   14. FEATURE-TO-OBSERVATION RATIO")
print("=" * 45)

feature_count = len(candidate_features)
observation_count = len(fraud_train)

print("Candidate features:", feature_count)
print("Training observations:", observation_count)
print("Observations per candidate feature:", round( observation_count / feature_count, 2))   


   14. FEATURE-TO-OBSERVATION RATIO
Candidate features: 19
Training observations: 1296675
Observations per candidate feature: 68246.05


In [22]:
print("\n" + "=" * 80)
print("\t\t\t15. NUMERICAL FEATURE VARIABILITY")
print("=" * 80)

numerical_candidate_features = [
    column
    for column in candidate_features
    if column in fraud_train.select_dtypes(
        include=np.number
    ).columns
]

if numerical_candidate_features:
    variability_summary = (fraud_train[numerical_candidate_features].describe().T)
    variability_summary["range"] = (variability_summary["max"] - variability_summary["min"])

    print(
        variability_summary[
            ["mean", "std", "min", "max", "range"]
        ].round(2)
    )
else:
    print("No numerical candidate features found.")


			15. NUMERICAL FEATURE VARIABILITY
                    mean          std           min           max        range
amt         7.035000e+01       160.32  1.000000e+00  2.894890e+04     28947.90
zip         4.880067e+04     26893.22  1.257000e+03  9.978300e+04     98526.00
lat         3.854000e+01         5.08  2.003000e+01  6.669000e+01        46.67
long       -9.023000e+01        13.76 -1.656700e+02 -6.795000e+01        97.72
city_pop    8.882444e+04    301956.36  2.300000e+01  2.906700e+06   2906677.00
unix_time   1.349244e+09  12841278.42  1.325376e+09  1.371817e+09  46440799.00
merch_lat   3.854000e+01         5.11  1.903000e+01  6.751000e+01        48.48
merch_long -9.023000e+01        13.77 -1.666700e+02 -6.695000e+01        99.72


In [23]:
print("\n" + "=" * 70)
print("\t\t16. EDA EVIDENCE FROM PROJECT 04")
print("=" * 70)

if fraud_eda is not None:
    print(f"\nProject 04 provided an EDA dataset containing: {len(fraud_eda)} observations.")
    if "is_fraud" in fraud_eda.columns:
        print("\nFraud observations identified in EDA dataset: ")
        print(fraud_eda["is_fraud"].value_counts())
    print("""
Project 04 demonstrated that the dataset can be investigated
across multiple dimensions including:

- transaction amount
- transaction category
- merchant
- geography
- transaction hour
- day of week
- customer age
- time-based patterns
""")
else:
    print("\nProject 04 analytical dataset unavailable.")


		16. EDA EVIDENCE FROM PROJECT 04

Project 04 provided an EDA dataset containing: 1852394 observations.

Fraud observations identified in EDA dataset: 
is_fraud
0    1842743
1       9651
Name: count, dtype: int64

Project 04 demonstrated that the dataset can be investigated
across multiple dimensions including:

- transaction amount
- transaction category
- merchant
- geography
- transaction hour
- day of week
- customer age
- time-based patterns



In [24]:
print("\n" + "=" * 68)
print("\t\t17. BUSINESS PROBLEM CLASSIFICATION")
print("=" * 68)

print("""
Business objective:
Predict whether a transaction is fraudulent or legitimate.

Target: is_fraud

Target values:
0 → Legitimate transaction
1 → Fraudulent transaction
""")
problem_type = "Supervised Binary Classification"

print("Recommended ML problem type:", problem_type)


		17. BUSINESS PROBLEM CLASSIFICATION

Business objective:
Predict whether a transaction is fraudulent or legitimate.

Target: is_fraud

Target values:
0 → Legitimate transaction
1 → Fraudulent transaction

Recommended ML problem type: Supervised Binary Classification


In [25]:
print("\n" + "=" * 82)
print("\t\t     18. SUITABLE MACHINE LEARNING APPROACHES")
print("=" * 82)

ml_approaches = pd.DataFrame({

    "approach": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
        "XGBoost / Similar Boosting Models"
    ],
     "suitability": [
        "Good baseline and interpretable",
        "Good for nonlinear relationships",
        "Strong general-purpose classification model",
        "Strong performance for structured/tabular data",
        "Potentially strong performance on tabular fraud data"
    ],

    "role": [
        "Baseline model",
        "Interpretable comparison",
        "Strong candidate",
        "Strong candidate",
        "Advanced candidate"
    ]
})
#print(ml_approaches.to_string(index=False))
ml_approaches


		     18. SUITABLE MACHINE LEARNING APPROACHES


,approach,suitability,role
0,Logistic Regression,Good baseline and interpretable,Baseline model
1,Decision Tree,Good for nonlinear relationships,Interpretable comparison
2,Random Forest,Strong general-purpose classification model,Strong candidate
3,Gradient Boosting,Strong performance for structured/tabular data,Strong candidate
4,XGBoost / Similar Boosting Models,Potentially strong performance on tabular frau...,Advanced candidate


In [26]:
print("\n" + "=" * 70)
print("\t     19. MACHINE LEARNING FEASIBILITY SCORECARD")
print("=" * 70)

scorecard = pd.DataFrame({

    "criterion": [
        "Sufficient observations",
        "Structured tabular data",
        "Clearly defined target",
        "Binary classification possible",
        "Missing values controlled",
        "Duplicate records controlled",
        "Multiple candidate features",
        "Business value from prediction",
        "Potential class imbalance",
        "Potential high-cardinality features",
        "Potential leakage concerns"
    ],
      "status": [
        "PASS" if total_rows >= 10000 else "REVIEW",
        "PASS" if len(fraud_train.columns) > 2 else "REVIEW",
        "PASS" if target in fraud_train.columns else "FAIL",
        "PASS" if invalid_target_values == set() else "FAIL",
        "PASS" if missing_values.empty else "REVIEW",
        "PASS" if duplicate_train == 0 else "REVIEW",
        "PASS" if len(candidate_features) >= 5 else "REVIEW",
        "PASS",
        "REVIEW",
        "REVIEW",
        "REVIEW"
    ],
     "reason": [
        size_assessment,
        "Dataset contains numerical and categorical variables.",
        "The is_fraud target is explicitly available.",
        "Target contains legitimate/fraudulent classes.",
        missing_assessment,
        duplicate_assessment,
        "Multiple transaction, customer and geographic attributes exist.",
        "Fraud prediction can support transaction-risk screening.",
        imbalance_assessment,
        "Some identifiers/categorical variables have high cardinality.",
        "Identifiers and temporal variables require careful feature selection."
    ]
})
scorecard
#print(scorecard.to_string(index=False))


	     19. MACHINE LEARNING FEASIBILITY SCORECARD


,criterion,status,reason
0,Sufficient observations,PASS,Very large dataset — highly suitable for ML ex...
1,Structured tabular data,PASS,Dataset contains numerical and categorical var...
2,Clearly defined target,PASS,The is_fraud target is explicitly available.
3,Binary classification possible,PASS,Target contains legitimate/fraudulent classes.
4,Missing values controlled,PASS,Missing-value quality is suitable for ML prepa...
5,Duplicate records controlled,PASS,No exact duplicate rows remain.
6,Multiple candidate features,PASS,"Multiple transaction, customer and geographic ..."
7,Business value from prediction,PASS,Fraud prediction can support transaction-risk ...
8,Potential class imbalance,REVIEW,Severe class imbalance detected. Special evalu...
9,Potential high-cardinality features,REVIEW,Some identifiers/categorical variables have hi...


In [27]:
print("\n" + "=" * 68)
print("20. FINAL MACHINE LEARNING FEASIBILITY DECISION")
print("=" * 68)

feasibility_decision = "FEASIBLE"
print("\nDecision:", feasibility_decision)

print("""
Reasoning:
1. The dataset contains a large number of transaction records.
2. The data is structured in a tabular format suitable for ML.
3. A clearly defined binary target, is_fraud, is available.
4. The target represents a meaningful business prediction problem.
5. Project 03 established a cleaned and validated data foundation.
6. Project 04 demonstrated meaningful fraud-related patterns.
7. Multiple numerical and categorical variables can potentially
   contribute predictive information.
8. Fraud detection provides clear business value because identifying
   suspicious transactions can support transaction screening and
   investigation.

However, ML implementation requires additional precautions:
- Class imbalance must be addressed.
- Accuracy alone should not be the primary evaluation metric.
- Precision, recall, F1-score, ROC-AUC and especially
  precision-recall analysis should be considered.
- Identifier columns should generally not be used as raw predictors.
- Categorical variables require suitable encoding.
- Temporal information must be handled without future-data leakage.
- The train/test strategy must reflect the real prediction scenario.
""")


20. FINAL MACHINE LEARNING FEASIBILITY DECISION

Decision: FEASIBLE

Reasoning:
1. The dataset contains a large number of transaction records.
2. The data is structured in a tabular format suitable for ML.
3. A clearly defined binary target, is_fraud, is available.
4. The target represents a meaningful business prediction problem.
5. Project 03 established a cleaned and validated data foundation.
6. Project 04 demonstrated meaningful fraud-related patterns.
7. Multiple numerical and categorical variables can potentially
   contribute predictive information.
8. Fraud detection provides clear business value because identifying
   suspicious transactions can support transaction screening and
   investigation.

However, ML implementation requires additional precautions:
- Class imbalance must be addressed.
- Accuracy alone should not be the primary evaluation metric.
- Precision, recall, F1-score, ROC-AUC and especially
  precision-recall analysis should be considered.
- Identifier column

In [28]:
print("\n" + "=" * 55)
print("21. RECOMMENDED MODELING STRATEGY FOR PROJECT 07")
print("=" * 55)

recommended_stratergy = pd.DataFrame({

    "stage": [
        "Data preparation",
        "Feature selection",
        "Categorical encoding",
        "Baseline",
        "Tree-based model",
        "Evaluation",
        "Business interpretation"
    ],
     "recommendation": [
        "Use Project 03 cleaned data.",
        "Remove target and unsuitable identifiers.",
        "Use appropriate encoding for categorical variables.",
        "Logistic Regression",
        "Random Forest / Gradient Boosting",
        "Precision, Recall, F1, ROC-AUC and PR-AUC",
        "Analyze fraud detection effectiveness and false alarms."
    ]
})
recommended_stratergy
#print(recommended_stratergy.to_string(index=False))


21. RECOMMENDED MODELING STRATEGY FOR PROJECT 07


,stage,recommendation
0,Data preparation,Use Project 03 cleaned data.
1,Feature selection,Remove target and unsuitable identifiers.
2,Categorical encoding,Use appropriate encoding for categorical varia...
3,Baseline,Logistic Regression
4,Tree-based model,Random Forest / Gradient Boosting
5,Evaluation,"Precision, Recall, F1, ROC-AUC and PR-AUC"
6,Business interpretation,Analyze fraud detection effectiveness and fals...


In [29]:
scorecard_path = (output_dir / "ml_feasibility_scorecard.csv")
scorecard.to_csv(scorecard_path, index=False)

feature_assessment_path = (output_dir / "feature_assessment.csv")
feature_assessment_df.to_csv(feature_assessment_path, index=False)

ml_approaches_path = (output_dir / "recommended_ml_approaches.csv")
ml_approaches.to_csv(ml_approaches_path, index=False)

class_summary_path = (output_dir / "class_distribution.csv")
class_summary.to_csv(class_summary_path)

recommended_stratergy_path = (output_dir / "project07_modeling_stratergy.csv")
recommended_stratergy.to_csv(recommended_stratergy_path, index=False)

In [30]:
final_report = pd.DataFrame({

    "assessment": [
        "Business problem",
        "ML feasibility",
        "ML problem type",
        "Training observations",
        "Testing observations",
        "Total observations",
        "Candidate features",
        "Training fraud rate",
        "Testing fraud rate",
        "Class imbalance",
        "Missing values",
        "Duplicate rows",
        "Recommended baseline",
        "Recommended advanced models"
    ],
    "result": [
        "Predict fraudulent vs legitimate transactions",
        feasibility_decision,
        problem_type,
        train_rows,
        test_rows,
        total_rows,
        len(candidate_features),
        round(train_fraud_rate, 2),
        round(test_fraud_rate, 2),
        imbalance_assessment,
        missing_assessment,
        duplicate_assessment,
        "Logistic Regression",
        "Random Forest / Gradient Boosting"
    ]
})
final_report_path = (output_dir / "ml_feasibility_report.csv")
final_report.to_csv(final_report_path, index=False)

In [31]:
print("\n" + "=" * 62)
print("\t\t    FINAL FEASIBILITY REPORT")
print("=" * 62)

#print(final_report.to_string(index=False))
final_report


		    FINAL FEASIBILITY REPORT


,assessment,result
0,Business problem,Predict fraudulent vs legitimate transactions
1,ML feasibility,FEASIBLE
2,ML problem type,Supervised Binary Classification
3,Training observations,1296675
4,Testing observations,555719
5,Total observations,1852394
6,Candidate features,19
7,Training fraud rate,0.58
8,Testing fraud rate,0.39
9,Class imbalance,Severe class imbalance detected. Special evalu...


In [32]:
print("\n" + "=" * 34)
print("PROJECT 06 OUTPUT FILES")
print("=" * 34)

for file in sorted(output_dir.iterdir()):
    print(" -", file.name)


PROJECT 06 OUTPUT FILES
 - class_distribution.csv
 - feature_assessment.csv
 - ml_feasibility_report.csv
 - ml_feasibility_scorecard.csv
 - project07_modeling_stratergy.csv
 - recommended_ml_approaches.csv


In [33]:
print("\n" + "=" * 68)
print("\t\t      PROJECT 06 COMPLETED")
print("=" * 68)

print("""
FINAL DECISION:

Machine Learning is FEASIBLE for this fraud-detection problem.

The dataset provides:
-Large-scale transaction observations
-Structured tabular data
- A clearly defined binary target
- Multiple numerical and categorical features
- Cleaned data from Project 03
- Fraud-related patterns identified in Project 04
- A meaningful business use case for prediction

The next stage is Project 07:
PREDICTIVE MODELING

Project 07 will convert this feasibility assessment
into an actual fraud-detection ML pipeline.
""")


		      PROJECT 06 COMPLETED

FINAL DECISION:

Machine Learning is FEASIBLE for this fraud-detection problem.

The dataset provides:
-Large-scale transaction observations
-Structured tabular data
- A clearly defined binary target
- Multiple numerical and categorical features
- Cleaned data from Project 03
- Fraud-related patterns identified in Project 04
- A meaningful business use case for prediction

The next stage is Project 07:
PREDICTIVE MODELING

Project 07 will convert this feasibility assessment
into an actual fraud-detection ML pipeline.

